# Tour API 관광지 상세 데이터 수집 및 정제

수집된 메인 관광지의 세부 정보(개요, 반려견 동반, 영업시간, 휴무일, 서브 이미지 등)를 크롤링하여 정제하는 파일입니다.

### 주요 작업:
1. **데이터 모델 구조화**:
   - **1:1 관계**: 개요(`detailCommon2`) 및 반려견 동반 정보(`detailPetTour2`)는 메인 장소 테이블에 직접 병합.
   - **1:N 관계**: 세부 소개(`detailIntro2`), 반복 정보(`detailInfo2`), 서브 이미지(`detailImage2`)는 별도 CSV 파일로 나누어 저장.
2. **점진적 수집 (캐싱)**: 일일 API 호출 한도 초과를 방지하기 위해 로컬 캐시를 활용하여 수집 완료된 항목은 스킵하고 이어서 가져옴.

## 1. 환경 설정 및 API 공통 함수 정의

In [1]:
!pip install requests pandas python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv

# .env 파일에서 API 키 로드 (상위 폴더 확인)
load_dotenv(dotenv_path="../../.env")
API_KEY = os.getenv("TOUR_API_DECODE_KEY")
BASE_URL = "https://apis.data.go.kr/B551011/KorService2"

print("API 키 로드 상태:", "성공" if API_KEY else "실패 (내부 .env 파일과 변수명을 확인하세요)")

def fetch_api_data(endpoint, base_url=BASE_URL, params=None):
    url = f"{base_url}/{endpoint}"
    default_params = {
        "serviceKey": API_KEY,
        "MobileOS": "ETC",
        "MobileApp": "RouteCheck",
        "_type": "json"
    }
    if params:
        default_params.update(params)
    
    max_retries = 3
    retry_delay = 2  # base delay in seconds
    
    for attempt in range(max_retries + 1):
        try:
            response = requests.get(url, params=default_params, timeout=10)
            
            # [HTTP 429: Too Many Requests 대응]
            if response.status_code == 429:
                if attempt < max_retries:
                    sleep_time = retry_delay * (2 ** attempt)
                    print(f"[WARNING] API 요청 제한(HTTP 429) 발생! {sleep_time}초 대기 후 재시도 ({attempt + 1}/{max_retries})...")
                    time.sleep(sleep_time)
                    continue
                else:
                    print(f"[ERROR] API 요청 제한(HTTP 429) 초과로 요청 최종 실패 ({endpoint})")
                    return pd.DataFrame()
            
            if response.status_code == 200:
                res_json = response.json()
                if 'response' in res_json:
                    header = res_json['response'].get('header', {})
                    result_code = header.get('resultCode')
                    if result_code != '0000':
                        # [공공데이터 포털 서버단 트래픽 제한 에러 대응 (예: '04' 또는 '22')]
                        if result_code in ['04', '22'] and attempt < max_retries:
                            sleep_time = retry_delay * (2 ** attempt)
                            print(f"[WARNING] API 트래픽 초과 에러 [{result_code}] 발생! {sleep_time}초 대기 후 재시도 ({attempt + 1}/{max_retries})...")
                            time.sleep(sleep_time)
                            continue
                        print(f"API 자체 에러 [{result_code}]: {header.get('resultMsg')} ({endpoint})")
                        return pd.DataFrame()
                    
                    body = res_json['response'].get('body', {})
                    items = body.get('items', {})
                    if not items or items == "":
                        return pd.DataFrame()
                    
                    if 'item' in items:
                        item_list = items['item']
                        if isinstance(item_list, dict):
                            item_list = [item_list]
                        return pd.DataFrame(item_list)
            else:
                if attempt < max_retries:
                    print(f"[WARNING] HTTP Error Code: {response.status_code} ({url}). {retry_delay}초 대기 후 재시도...")
                    time.sleep(retry_delay)
                    continue
                else:
                    print(f"HTTP Error Code: {response.status_code} ({url})")
                    return pd.DataFrame()
        except Exception as e:
            if attempt < max_retries:
                print(f"[WARNING] 네트워크 또는 데이터 파싱 실패 ({endpoint}): {e}. {retry_delay}초 대기 후 재시도...")
                time.sleep(retry_delay)
                continue
            else:
                print(f"네트워크 또는 데이터 파싱 최종 실패 ({endpoint}): {e}")
                return pd.DataFrame()
                
    return pd.DataFrame()

API 키 로드 상태: 성공


## 2. 메인 데이터 로드 및 크롤링 대상 식별

In [3]:
CRAWL_LIMIT = 1000 # 이번 회차에 수집할 타겟의 개수 (전체 수집을 원하면 None으로 지정)
os.makedirs("../data", exist_ok=True)

# 파일 저장 경로
path_main_csv = "../data/관광정보_메인_장소_데이터.csv"
path_detail_common = "../data/관광정보_상세_공통_raw.csv"
path_detail_intro = "../data/관광정보_상세_소개_raw.csv"
path_detail_info = "../data/관광정보_상세_반복_raw.csv"
path_detail_image = "../data/관광정보_상세_이미지_raw.csv"
path_detail_pet = "../data/관광정보_상세_반려동물_raw.csv"

def load_cache_or_empty(file_path, columns):
    if os.path.exists(file_path):
        try:
            df = pd.read_csv(file_path, encoding='utf-8-sig')
            df['contentid'] = df['contentid'].astype(str).str.strip()
            return df
        except Exception as e:
            print(f"{file_path} 캐시 로드 실패: {e}")
    return pd.DataFrame(columns=columns)

# 1. 메인 기본 장소 데이터 로드
if os.path.exists(path_main_csv):
    df_main_base = pd.read_csv(path_main_csv, encoding='utf-8-sig')
    df_main_base['contentid'] = df_main_base['contentid'].astype(str).str.strip()
    all_contentids = df_main_base['contentid'].tolist()
    print(f"기본 메인 장소 데이터 로드 완료! 총 개수: {len(df_main_base)}개")
else:
    raise FileNotFoundError(f"{path_main_csv} 파일이 존재하지 않습니다. 먼저 메인 데이터 수집 노트북을 실행하세요.")

# 2. 캐시 로드
cache_common = load_cache_or_empty(path_detail_common, ['contentid', 'overview', 'homepage'])
cache_intro = load_cache_or_empty(path_detail_intro, ['contentid'])
cache_info = load_cache_or_empty(path_detail_info, ['contentid'])
cache_image = load_cache_or_empty(path_detail_image, ['contentid'])
cache_pet = load_cache_or_empty(path_detail_pet, ['contentid'])

# 3. 크롤링되지 않은 대상 필터링 (detailCommon2 수집 여부 기준)
crawled_common_ids = set(cache_common['contentid'].tolist())
todo_ids = [cid for cid in all_contentids if cid not in crawled_common_ids]

print(f"기존 수집 완료 개수: {len(crawled_common_ids)}개")
print(f"미수집 개수: {len(todo_ids)}개")

target_ids = todo_ids[:CRAWL_LIMIT] if CRAWL_LIMIT else todo_ids
print(f"이번 회차 수집 개수: {len(target_ids)}개")

기본 메인 장소 데이터 로드 완료! 총 개수: 50677개
기존 수집 완료 개수: 0개
미수집 개수: 50677개
이번 회차 수집 개수: 1000개


## 3. 상세 API 크롤링 및 로컬 캐싱 수집

### 4-1. 공통 상세 정보 조회 (`detailCommon2`) - 1:1

In [4]:
if len(target_ids) > 0:
    print("--- 공통 상세 정보 조회 (detailCommon2) 시작 ---")
    new_common_list = []
    
    for i, cid in enumerate(target_ids):
        params = {
            "contentId": cid,
            "defaultYN": "Y",
            "overviewYN": "Y"
        }
        df_chunk = fetch_api_data("detailCommon2", params=params)
        
        if not df_chunk.empty:
            overview_val = df_chunk['overview'].iloc[0] if 'overview' in df_chunk.columns else ""
            homepage_val = df_chunk['homepage'].iloc[0] if 'homepage' in df_chunk.columns else ""
            new_common_list.append({
                "contentid": cid,
                "overview": overview_val,
                "homepage": homepage_val
            })
        
        if (i+1) % 20 == 0:
            print(f"진행률: {i+1}/{len(target_ids)} 완료")
        time.sleep(0.1)
    
    if new_common_list:
        new_common_df = pd.DataFrame(new_common_list)
        cache_common = pd.concat([cache_common, new_common_df], ignore_index=True).drop_duplicates(subset=['contentid'], keep='last')
        cache_common.to_csv(path_detail_common, index=False, encoding='utf-8-sig')
        print(f"공통 상세 크롤링 완료 및 저장! 누적 수집된 데이터 크기: {cache_common.shape}")
else:
    print("새로 크롤링할 타겟이 없습니다.")

--- 공통 상세 정보 조회 (detailCommon2) 시작 ---
진행률: 20/1000 완료
진행률: 40/1000 완료


KeyboardInterrupt: 

### 4-2. 반려동물 동반여행 정보 조회 (`detailPetTour2`) - 1:1

In [ ]:
if len(target_ids) > 0:
    print("--- 반려동물 동반 정보 조회 (detailPetTour2) 시작 ---")
    new_pet_list = []
    
    for i, cid in enumerate(target_ids):
        params = {
            "contentId": cid
        }
        df_chunk = fetch_api_data("detailPetTour2", params=params)
        if not df_chunk.empty:
            pet_dict = df_chunk.iloc[0].to_dict()
            pet_dict['contentid'] = cid
            new_pet_list.append(pet_dict)
        else:
            new_pet_list.append({"contentid": cid})
        
        if (i+1) % 20 == 0:
            print(f"진행률: {i+1}/{len(target_ids)} 완료")
        time.sleep(0.1)
        
    if new_pet_list:
        new_pet_df = pd.DataFrame(new_pet_list)
        cache_pet = pd.concat([cache_pet, new_pet_df], ignore_index=True).drop_duplicates(subset=['contentid'], keep='last')
        cache_pet.to_csv(path_detail_pet, index=False, encoding='utf-8-sig')
        print(f"반려동물 정보 크롤링 완료 및 저장! 누적 수집된 데이터 크기: {cache_pet.shape}")

### 4-3. 소개정보 조회 (`detailIntro2`) - 1:1 dynamic

In [ ]:
if len(target_ids) > 0:
    print("--- 소개 정보 조회 (detailIntro2) 시작 ---")
    new_intro_list = []
    
    # contenttypename을 다시 API 조회용 contenttypeid 코드로 역매핑
    type_name_to_id = {
        '관광지': '12', '문화시설': '14', '축제/공연/행사': '15', '여행 코스': '25',
        '레포츠': '28', '숙박': '32', '쇼핑': '38', '음식점': '39'
    }
    df_target = df_main_base[df_main_base['contentid'].isin(target_ids)][['contentid', 'contenttypename']].drop_duplicates().copy()
    df_target['contenttypeid'] = df_target['contenttypename'].map(type_name_to_id)
    
    for i, row in df_target.iterrows():
        cid = row['contentid']
        ctid = row['contenttypeid']
        if not ctid or pd.isna(ctid):
            continue
        params = {
            "contentId": cid,
            "contentTypeId": ctid
        }
        df_chunk = fetch_api_data("detailIntro2", params=params)
        if not df_chunk.empty:
            intro_dict = df_chunk.iloc[0].to_dict()
            intro_dict['contentid'] = cid
            new_intro_list.append(intro_dict)
        else:
            new_intro_list.append({"contentid": cid, "contenttypeid": ctid})
            
        if (i+1) % 20 == 0:
            print(f"진행률: {i+1}/{len(df_target)} 완료")
        time.sleep(0.1)
        
    if new_intro_list:
        new_intro_df = pd.DataFrame(new_intro_list)
        cache_intro = pd.concat([cache_intro, new_intro_df], ignore_index=True).drop_duplicates(subset=['contentid'], keep='last')
        cache_intro.to_csv(path_detail_intro, index=False, encoding='utf-8-sig')
        print(f"소개 정보 크롤링 완료 및 저장! 누적 수집된 데이터 크기: {cache_intro.shape}")

### 4-4. 반복정보 조회 (`detailInfo2`) - 1:N

In [ ]:
if len(target_ids) > 0:
    print("--- 반복 정보 조회 (detailInfo2) 시작 ---")
    new_info_list = []
    
    # contenttypename을 다시 API 조회용 contenttypeid 코드로 역매핑
    type_name_to_id = {
        '관광지': '12', '문화시설': '14', '축제/공연/행사': '15', '여행 코스': '25',
        '레포츠': '28', '숙박': '32', '쇼핑': '38', '음식점': '39'
    }
    df_target = df_main_base[df_main_base['contentid'].isin(target_ids)][['contentid', 'contenttypename']].drop_duplicates().copy()
    df_target['contenttypeid'] = df_target['contenttypename'].map(type_name_to_id)
    
    for i, row in df_target.iterrows():
        cid = row['contentid']
        ctid = row['contenttypeid']
        if not ctid or pd.isna(ctid):
            continue
        params = {
            "contentId": cid,
            "contentTypeId": ctid
        }
        df_chunk = fetch_api_data("detailInfo2", params=params)
        if not df_chunk.empty:
            for _, info_row in df_chunk.iterrows():
                info_dict = info_row.to_dict()
                info_dict['contentid'] = cid
                new_info_list.append(info_dict)
                
        if (i+1) % 20 == 0:
            print(f"진행률: {i+1}/{len(df_target)} 완료")
        time.sleep(0.1)
        
    if new_info_list:
        new_info_df = pd.DataFrame(new_info_list)
        cache_info = pd.concat([cache_info, new_info_df], ignore_index=True).drop_duplicates()
        cache_info.to_csv(path_detail_info, index=False, encoding='utf-8-sig')
        print(f"반복 정보 크롤링 완료 및 저장! 누적 수집된 데이터 크기: {cache_info.shape}")

### 4-5. 이미지정보 조회 (`detailImage2`) - 1:N

In [ ]:
if len(target_ids) > 0:
    print("--- 이미지 정보 조회 (detailImage2) 시작 ---")
    new_image_list = []
    
    for i, cid in enumerate(target_ids):
        params = {
            "contentId": cid,
            "imageYN": "Y",
            "subImageYN": "Y"
        }
        df_chunk = fetch_api_data("detailImage2", params=params)
        if not df_chunk.empty:
            for _, img_row in df_chunk.iterrows():
                img_dict = img_row.to_dict()
                img_dict['contentid'] = cid
                new_image_list.append(img_dict)
                
        if (i+1) % 20 == 0:
            print(f"진행률: {i+1}/{len(target_ids)} 완료")
        time.sleep(0.1)
        
    if new_image_list:
        new_image_df = pd.DataFrame(new_image_list)
        cache_image = pd.concat([cache_image, new_image_df], ignore_index=True).drop_duplicates()
        cache_image.to_csv(path_detail_image, index=False, encoding='utf-8-sig')
        print(f"이미지 정보 크롤링 완료 및 저장! 누적 수집된 데이터 크기: {cache_image.shape}")

## 4. 최종 데이터셋 병합 및 분리 저장 (CSV 단일화)

In [ ]:
print("\n--- 최종 데이터셋 병합 및 저장 ---")

# 1. 메인 기본 장소 데이터 다시 읽기
df_main_export = pd.read_csv(path_main_csv, encoding='utf-8-sig')
df_main_export['contentid'] = df_main_export['contentid'].astype(str).str.strip()

# 2. 1:1 관계 상세데이터 병합 (Common + Pet)
# 병합 시 중복 컬럼 생김 방지 위해 기존 메인 데이터에 컬럼이 있으면 제거 후 병합
if not cache_common.empty:
    cache_common['contentid'] = cache_common['contentid'].astype(str).str.strip()
    # overview, homepage 컬럼이 이미 메인에 존재한다면 제거
    cols_to_remove = [c for c in ['overview', 'homepage'] if c in df_main_export.columns]
    df_main_export.drop(columns=cols_to_remove, inplace=True, errors='ignore')
    df_main_export = pd.merge(df_main_export, cache_common[['contentid', 'overview', 'homepage']], on='contentid', how='left')
    
if not cache_pet.empty:
    cache_pet['contentid'] = cache_pet['contentid'].astype(str).str.strip()
    pet_cols = ['contentid', 'acmpyTypeCd', 'acmpyPsblCpam', 'acmpyNeedMtr', 'etcAcmpyInfo']
    existing_pet_cols = [c for c in pet_cols if c in cache_pet.columns]
    cols_to_remove = [c for c in existing_pet_cols if c != 'contentid' and c in df_main_export.columns]
    df_main_export.drop(columns=cols_to_remove, inplace=True, errors='ignore')
    df_main_export = pd.merge(df_main_export, cache_pet[existing_pet_cols], on='contentid', how='left')

# 3. 메인 컬럼 순서 재정렬 (title, contentid, contenttypename을 맨 앞으로)
front_cols = ['title', 'contentid', 'contenttypename']
remaining_cols = [col for col in df_main_export.columns if col not in front_cols]
df_main_export = df_main_export[front_cols + remaining_cols]

# 4. 메인 데이터셋 및 상세 분리 데이터셋 CSV 저장
path_intro_csv = "../data/관광정보_소개정보.csv"
path_info_csv = "../data/관광정보_세부반복정보.csv"
path_image_csv = "../data/관광정보_추가이미지.csv"

df_main_export.to_csv(path_main_csv, index=False, encoding='utf-8-sig')
print(f"✅ [메인 저장 완료] {path_main_csv} ({len(df_main_export)}행)")

if not cache_intro.empty:
    cache_intro.to_csv(path_intro_csv, index=False, encoding='utf-8-sig')
    print(f"✅ [소개 저장 완료] {path_intro_csv} ({len(cache_intro)}행)")

if not cache_info.empty:
    cache_info.to_csv(path_info_csv, index=False, encoding='utf-8-sig')
    print(f"✅ [반복 저장 완료] {path_info_csv} ({len(cache_info)}행)")

if not cache_image.empty:
    cache_image.to_csv(path_image_csv, index=False, encoding='utf-8-sig')
    print(f"✅ [이미지 저장 완료] {path_image_csv} ({len(cache_image)}행)")

# 5. 기존 생성된 JSON 파일 삭제 (CSV 단일화 정책)
for file_name in ["관광정보_메인_장소_데이터.json", "관광정보_소개정보.json", "관광정보_세부반복정보.json", "관광정보_추가이미지.json"]:
    json_file_path = os.path.join("../data", file_name)
    if os.path.exists(json_file_path):
        try:
            os.remove(json_file_path)
            print(f"[SUCCEE] 기존 JSON 파일 삭제 완료: {json_file_path}")
        except Exception as e:
            print(f"[ERROR] {json_file_path} 삭제 실패: {e}")